 # Phase 1 : exploration du jeu de candidats

 Ouvrir dans VSCode (cellules `# %%`), lancer depuis la racine du dépôt.

 Prérequis : `python scripts/build_dataset.py` a produit `data/processed/candidates_v1.csv.gz`.

 Prototype de recherche : aucun diagnostic clinique.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import matplotlib.pyplot as plt
import pandas as pd

from src.features.builder import FAMILIES, feature_columns

df = pd.read_csv("data/processed/candidates_v1.csv.gz")
train = df[df["split"] == "train"]
OUT = Path("reports/phase_1")
OUT.mkdir(parents=True, exist_ok=True)


 ## 1. Taille, prévalence, candidats par image

In [ ]:
print(df.groupby("split")["label"].agg(n_candidats="size", n_parasites="sum", prevalence="mean").round(3))
print("\ncandidats par image :\n", df.groupby("image_id").size().describe().round(1))
print("\nfeatures par famille :", {f: len(feature_columns(df, (f,))) for f in FAMILIES})


 ## 2. Pouvoir séparateur de chaque feature (AUC univariée, TRAIN uniquement)

 AUC = probabilité qu'un candidat « parasite » ait une valeur plus grande qu'un « non-parasite ».

 0,5 = aucune information ; loin de 0,5 (dans un sens ou l'autre) = informatif.

In [ ]:
def auc(x: pd.Series, y: pd.Series) -> float:
    n1 = int(y.sum())
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return float("nan")
    r = x.rank()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


cols = [c for c in feature_columns(df, FAMILIES) if not c.startswith("hog_")]  # HOG : 324 colonnes, vues par ablation
scores = pd.Series({c: auc(train[c], train["label"]) for c in cols})
top = (scores - 0.5).abs().sort_values(ascending=False).index[:15]
print(scores[top].round(3))


 ## 3. Variables de contexte `qual_` : indices d'un effet « image » (appareil, lame, éclairage)

 Une AUC éloignée de 0,5 sur une variable qui décrit l'image entière, pas le candidat,

 signale que l'étiquette dépend de l'image : à surveiller en Phase 2 et 3.

In [ ]:
print(scores[[c for c in cols if c.startswith("qual_")]].round(3))


 ## 4. Distributions des 6 features les plus séparatrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, c in zip(axes.ravel(), top[:6]):
    for lab, name in ((0, "non-parasite"), (1, "parasite")):
        ax.hist(train.loc[train["label"] == lab, c], bins=40, alpha=0.55, density=True, label=name)
    ax.set_title(c, fontsize=9)
axes[0, 0].legend()
fig.suptitle("Distributions par classe (train)")
fig.tight_layout()
fig.savefig(OUT / "features_top6_distributions.png", dpi=120)
plt.show()


 ## 5. Anomalies à noter dans la note de synthèse

 - nombre de features constantes ; valeurs aberrantes ; images sans candidat ; déséquilibre.

In [ ]:
const = [c for c in feature_columns(df, FAMILIES) if train[c].nunique() <= 1]
print("features constantes sur le train :", const)
print("images sans aucun candidat (dans cette table) : voir reports/phase_1/candidate_recall.json")